# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: What recurring performance archetypes exist across FlyRank's content inventory, based on observable search and engagement signals?

The decision this supports: Given limited reviewer capacity, which broad strategy — protect, improve, rewrite, merge, prune, or monitor — should be applied to a page, before any individual human review begins. This is a triage-support question, not a prediction or diagnosis question.

Who acts on it: A content/SEO reviewer working through a large inventory who needs a starting map of "kinds of pages we have," rather than reading tens of thousands of rows individually.

Why this is framed as unsupervised, not supervised: No ground-truth label exists for "content archetype" — inventing one (e.g., borrowing FlyRank's own internal health_score or action_type) would risk building a model that just re-learns an existing product decision rather than discovering real structure in the data. This reasoning was established early (w02) and held throughout: every feature and every validation choice in this project was checked against it.

Cost of a wrong call: Lower-stakes than a binary decline/growth prediction, because a cluster assignment is a lens, not a verdict — no page loses traffic because it was mis-clustered. The real cost is wasted reviewer time and eroded trust in the system: if archetypes are named or interpreted carelessly, reviewers stop trusting the labels, and pages that genuinely need attention get deprioritized.

In [1]:
research_question = {
    "question": "What recurring performance archetypes exist across the content inventory, "
                 "based on observable search and engagement signals?",
    "decision_supported": "Triage priority (protect/improve/rewrite/merge/prune/monitor) "
                           "before individual human review",
    "task_type": "Unsupervised clustering — no label exists or was invented",
    "cost_of_wrong_call": "Low-to-moderate — wasted reviewer time and reduced trust in the "
                           "system, not a page-level harm, since output is decision-support only",
}
print(research_question["question"])
print("\nDecision supported:", research_question["decision_supported"])
print("Task type:", research_question["task_type"])

What recurring performance archetypes exist across the content inventory, based on observable search and engagement signals?

Decision supported: Triage priority (protect/improve/rewrite/merge/prune/monitor) before individual human review
Task type: Unsupervised clustering — no label exists or was invented


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Source: FlyRank internship warehouse (Hugging Face, FlyRank/internship-warehouse), queried live via DuckDB against Parquet files.

Tables used: fact_content_daily_performance (partition month=2026-03) joined to dim_content, both on client_hash_id + content_hash_id.

Date window: March 1–31, 2026 — a single, frozen mid-panel month. Deliberately not fact_content_daily_performance_sample, which is exactly June 2026, the panel's final month and the natural outcome window for any past→future label; using it during model development would risk contaminating a sealed test period.

What was excluded, and why — public-safe:

*  Any product-decision field (health_score, priority_score, action_type, recommendation, cluster, archetype) — excluded to prevent the model from re-learning an existing business rule instead of discovering real structure.

* GA4-sourced engagement fields — technically available, but severely sparse (4.21% of rows) and excluded from the core feature set for that reason (kept only as an optional post-cluster diagnostic).

* Rows outside GSC tracking coverage — the modeling population is necessarily the ~34% of the full inventory with real March search-visibility data; content with no GSC coverage that month is invisible to this analysis by construction, not by choice.

*  All client names, real URLs, and search query text — only hashed IDs and aggregate metrics are used or reported anywhere in this project.

In [2]:
import os, duckdb, numpy as np, pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# --- Window + grain verification ---
window_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT report_date) AS distinct_days
    FROM read_parquet('{TABLE}')
""").df()
print("Window verification:")
print(window_check.to_string(index=False))

grain_check = con.sql(f"""
    SELECT COUNT(*) AS duplicate_groups FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM read_parquet('{TABLE}') GROUP BY 1,2,3 HAVING COUNT(*) > 1
    )
""").df()
print("\nDuplicate grain groups (should be 0):", grain_check['duplicate_groups'].iloc[0])

# --- Availability rates ---
avail = con.sql(f"""
    SELECT COUNT(*) AS total,
           SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_avail,
           SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_avail
    FROM read_parquet('{TABLE}')
""").df().iloc[0]
gsc_pct = round(avail['gsc_avail'] / avail['total'] * 100, 2)
ga4_pct = round(avail['ga4_avail'] / avail['total'] * 100, 2)
print(f"\nGSC availability: {gsc_pct}%  |  GA4 availability: {ga4_pct}%")

# --- Full inventory vs. modeling population ---
dim_total = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{DIM}')").df().iloc[0]['n']

modeling_pop = con.sql(f"""
    SELECT COUNT(DISTINCT (client_hash_id, content_hash_id)) AS n
    FROM read_parquet('{TABLE}') WHERE gsc_data_available = TRUE
""").df().iloc[0]['n']

coverage_pct = round(modeling_pop / dim_total * 100, 2)
print(f"\nFull inventory (dim_content): {dim_total:,}")
print(f"Modeling population (GSC-available, this window): {modeling_pop:,}")
print(f"Coverage: {coverage_pct}% of full inventory")

data_summary = {
    "source": "FlyRank internship warehouse (Hugging Face)",
    "tables": ["fact_content_daily_performance (month=2026-03)", "dim_content"],
    "window": f"{window_check['min_date'].iloc[0]} to {window_check['max_date'].iloc[0]}",
    "raw_rows": int(window_check['total_rows'].iloc[0]),
    "distinct_days": int(window_check['distinct_days'].iloc[0]),
    "duplicate_grain_groups": int(grain_check['duplicate_groups'].iloc[0]),
    "gsc_availability_pct": gsc_pct,
    "ga4_availability_pct": ga4_pct,
    "full_inventory": int(dim_total),
    "modeling_population": int(modeling_pop),
    "coverage_pct_of_inventory": coverage_pct,
}
print("\n", data_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Window verification:
 total_rows   min_date   max_date  distinct_days
    9841378 2026-03-01 2026-03-31             31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Duplicate grain groups (should be 0): 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


GSC availability: 36.69%  |  GA4 availability: 4.21%

Full inventory (dim_content): 519,606
Modeling population (GSC-available, this window): 176,738
Coverage: 34.01% of full inventory

 {'source': 'FlyRank internship warehouse (Hugging Face)', 'tables': ['fact_content_daily_performance (month=2026-03)', 'dim_content'], 'window': '2026-03-01 00:00:00 to 2026-03-31 00:00:00', 'raw_rows': 9841378, 'distinct_days': 31, 'duplicate_grain_groups': 0, 'gsc_availability_pct': np.float64(36.69), 'ga4_availability_pct': np.float64(4.21), 'full_inventory': 519606, 'modeling_population': 176738, 'coverage_pct_of_inventory': np.float64(34.01)}


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Features (5 conceptual, 7 in the final model matrix): gsc_impressions, gsc_avg_position, gsc_clicks, content_age_days, word_count, plus two missingness/placeholder flags derived from the first two.

Preprocessing:



* gsc_avg_position = 0 (0.81% of rows) means no rank data, not a real position — converted to missing, imputed with the median of real values, and flagged (avg_position_missing_or_zero).

*  gsc_impressions and gsc_clicks are heavily right-skewed — log-transformed (log1p) before scaling to prevent a few extreme pages from dominating distance calculations.

* word_count was missing for 31.30% of rows — imputed with the population median and flagged (word_count_missing), so the model can distinguish "short content" from "unknown content length" rather than conflating them.

* All 7 features scaled with StandardScaler, fit on the training split only, then applied to validation — no leakage through the preprocessing step itself.


Label definition: None. This is unsupervised clustering by design (Section 1) — no proxy label was invented at any stage.

Baseline: A transparent, hand-coded rule (TITLE_META_CTR_FIX) — flag a page if it has ≥100 impressions, a usable (non-zero) position, and CTR at least 30% below the pooled expected CTR for its position tier. Built on two signals checked first: staleness (OPPOSITE — a real negative result, excluded) and CTR-vs-position (CONFIRMED — became the rule's basis).

Validation design: Client-grouped split (GroupShuffleSplit, 75/25) — pages from the same client plausibly share a CMS template, editorial convention, and tracking setup, so a naive random split risks letting a client's "house style" leak between train and validation.

Leakage checks: (1) static check confirming no product-decision fields ever entered the feature set; (2) a deliberate leak-trap — a cluster-derived proxy feature appended to the honest feature set inflates silhouette artificially, then removed, keeping only the honest number.


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

# --- Build modeling population ---
raw = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}') WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions, SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
        FROM scoped GROUP BY client_hash_id, content_hash_id
    )
    SELECT a.*, d.content_created_date, d.word_count
    FROM agg a LEFT JOIN read_parquet('{DIM}') d
      ON a.client_hash_id = d.client_hash_id AND a.content_hash_id = d.content_hash_id
    ORDER BY a.client_hash_id, a.content_hash_id
""").df()

df = raw.copy()
df['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(df['content_created_date'])).dt.days
df['avg_position_missing_or_zero'] = (df['gsc_avg_position'] == 0).astype(int)
df['avg_position_clean'] = df['gsc_avg_position'].replace(0, np.nan)
avg_pos_median = df.loc[df['avg_position_clean'].notna(), 'avg_position_clean'].median()
df['avg_position_clean'] = df['avg_position_clean'].fillna(avg_pos_median)
df['log_gsc_impressions'] = np.log1p(df['gsc_impressions'])
df['log_gsc_clicks'] = np.log1p(df['gsc_clicks'])
df['word_count_missing'] = df['word_count'].isna().astype(int)
word_count_median = df['word_count'].median()
df['word_count'] = df['word_count'].fillna(word_count_median)

MODEL_FEATURES = ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks',
                   'content_age_days', 'word_count', 'avg_position_missing_or_zero', 'word_count_missing']

# --- Leakage check ---
forbidden_fields = {"health_score", "priority_score", "action_type", "recommended_action",
                     "recommendation", "cluster", "archetype", "label", "target"}
print("Forbidden fields in feature set:", forbidden_fields.intersection(set(MODEL_FEATURES)) or "NONE — clean")

# --- Client-grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['client_hash_id'].values))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[MODEL_FEATURES])
X_val = scaler.transform(val_df[MODEL_FEATURES])

print(f"\nTrain: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Val: {len(val_df)} rows, {val_df['client_hash_id'].nunique()} clients")
print("Client overlap:", len(set(train_df['client_hash_id']) & set(val_df['client_hash_id'])))

# --- Leak-trap demonstration ---
from sklearn.preprocessing import OneHotEncoder
km_honest = KMeans(n_clusters=4, random_state=42, n_init=30)
labels_honest = km_honest.fit_predict(X_train)
score_honest = silhouette_score(X_train, labels_honest, sample_size=20000, random_state=42)

leak_onehot = OneHotEncoder(sparse_output=False).fit_transform(labels_honest.reshape(-1, 1))
X_leaky = np.hstack([X_train, leak_onehot])
score_leaky = silhouette_score(X_leaky, labels_honest, sample_size=20000, random_state=42)

print(f"\nHonest silhouette: {score_honest:.4f}")
print(f"Leaky silhouette (cluster-derived proxy appended): {score_leaky:.4f}")
print(f"Jump: +{score_leaky - score_honest:.4f} — leak removed, honest number kept")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Forbidden fields in feature set: NONE — clean

Train: 133474 rows, 35 clients
Val: 43264 rows, 12 clients
Client overlap: 0

Honest silhouette: 0.2944
Leaky silhouette (cluster-derived proxy appended): 0.3541
Jump: +0.0597 — leak removed, honest number kept


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Model vs. baseline, same population, honest split, real numbers:

# Model Validation Summary

## 1. Week 4 baseline

- **Population:** eligible rows (impressions≥100, position>0)
- **Output:** `TITLE_META_CTR_FIX` queue
- **Metric:** coverage
- **Result:** 61,267 flagged

---

## 2. K-Means (k=4)

- **Population:** 176,738 rows, client-grouped split
- **Output:** 4 clusters
- **Metric:** validation silhouette
- **Result:** 0.3568

---

## 3. Baseline overlay

- **Population:** same rows
- **Output:** lift by cluster
- **Metric:** max/min lift
- **Result:** 1.197 / 0.000

---

## 4. Stability check

- **Population:** same rows
- **Output:** seed consistency (5 seeds)
- **Metric:** mean ARI
- **Result:** ≥0.996

---

## 5. Split honesty check

- **Population:** same model, same data
- **Output:** naive vs. grouped split
- **Metric:** silhouette gap
- **Result:** 0.2962 (naive, 45/47 client overlap) vs. 0.3568 (grouped, 0 overlap)

In [4]:
# --- k-selection sweep ---
k_rows = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    tl = km.fit_predict(X_train)
    vl = km.predict(X_val)
    k_rows.append({
        'k': k,
        'val_silhouette': round(silhouette_score(X_val, vl, sample_size=20000, random_state=42), 4),
        'val_davies_bouldin': round(davies_bouldin_score(X_val, vl), 4),
    })
k_df = pd.DataFrame(k_rows)
print("K-selection:\n", k_df.to_string(index=False))

# --- Final model, k=4 ---
FINAL_K = 4
final_km = KMeans(n_clusters=FINAL_K, random_state=42, n_init=30)
train_labels = final_km.fit_predict(X_train)
val_labels = final_km.predict(X_val)
train_df['cluster'] = train_labels
val_df['cluster'] = val_labels
full_labeled = pd.concat([train_df.assign(split='train'), val_df.assign(split='val')], ignore_index=True)

print(f"\nSelected k=4 | val silhouette: {k_df[k_df.k==4]['val_silhouette'].iloc[0]} "
      f"| val DB: {k_df[k_df.k==4]['val_davies_bouldin'].iloc[0]}")

# --- Seed stability ---
seeds = [7, 13, 29, 42, 101]
seed_labels = {s: KMeans(n_clusters=FINAL_K, random_state=s, n_init=30).fit_predict(X_train) for s in seeds}
aris = [adjusted_rand_score(seed_labels[seeds[i]], seed_labels[seeds[j]])
        for i in range(len(seeds)) for j in range(i+1, len(seeds))]
print(f"Seed stability mean ARI: {np.mean(aris):.4f}")

# --- Naive vs grouped split comparison ---
train_naive, val_naive = train_test_split(df, test_size=0.25, random_state=42)
scaler_n = StandardScaler()
Xn_train = scaler_n.fit_transform(train_naive[MODEL_FEATURES])
Xn_val = scaler_n.transform(val_naive[MODEL_FEATURES])
km_naive = KMeans(n_clusters=4, random_state=42, n_init=30)
km_naive.fit(Xn_train)
naive_val_labels = km_naive.predict(Xn_val)
naive_sil = silhouette_score(Xn_val, naive_val_labels, sample_size=20000, random_state=42)
naive_overlap = len(set(train_naive['client_hash_id']) & set(val_naive['client_hash_id']))
print(f"\nNaive split: silhouette={naive_sil:.4f}, client overlap={naive_overlap}/{df['client_hash_id'].nunique()}")

# --- Baseline rebuild + overlay ---
baseline_base = full_labeled[(full_labeled['gsc_impressions'] >= 100) & (full_labeled['gsc_avg_position'] > 0)].copy()
baseline_base['ctr'] = baseline_base['gsc_clicks'] / baseline_base['gsc_impressions']
def position_tier(pos):
    if pos <= 3: return 'pos_1_3'
    elif pos <= 10: return 'pos_4_10'
    elif pos <= 20: return 'pos_11_20'
    else: return 'pos_21_plus'
baseline_base['position_tier'] = baseline_base['gsc_avg_position'].apply(position_tier)
tier_ctr = baseline_base.groupby('position_tier').apply(
    lambda g: g['gsc_clicks'].sum() / g['gsc_impressions'].sum(), include_groups=False).to_dict()
baseline_base['expected_ctr'] = baseline_base['position_tier'].map(tier_ctr)
baseline_base['ctr_gap_pct'] = (baseline_base['expected_ctr'] - baseline_base['ctr']).clip(lower=0) / baseline_base['expected_ctr']
baseline_base['baseline_flag'] = (baseline_base['ctr_gap_pct'] >= 0.30).astype(int)

print(f"\nBaseline queue: {baseline_base['baseline_flag'].sum()} rows")

lift = baseline_base.groupby('cluster').agg(rows=('cluster','size'), flagged=('baseline_flag','sum')).reset_index()
lift['flag_rate'] = lift['flagged'] / lift['rows']
lift['lift'] = lift['flag_rate'] / baseline_base['baseline_flag'].mean()
print("\nBaseline lift by cluster:\n", lift.to_string(index=False))

K-selection:
  k  val_silhouette  val_davies_bouldin
 3          0.3485              1.1340
 4          0.3568              0.9087
 5          0.3416              1.1002
 6          0.3395              0.9820
 7          0.3314              1.0163
 8          0.3270              0.9931

Selected k=4 | val silhouette: 0.3568 | val DB: 0.9087
Seed stability mean ARI: 0.9981

Naive split: silhouette=0.2962, client overlap=45/47

Baseline queue: 61267 rows

Baseline lift by cluster:
  cluster  rows  flagged  flag_rate     lift
       0 25856    20234   0.782565 1.295709
       1 48555    20224   0.416517 0.689636
       2 27030    20809   0.769848 1.274653


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
